In [ ]:
# pip install dotenv

In [13]:
import os
from dotenv import load_dotenv

In [19]:
# Import api_key
load_dotenv()

API_KEY = os.getenv('API_KEY')

## **Pencarian Koordinat Kantor Wali Nagari**

In [ ]:
import pandas as pd
import googlemaps
import time

# 1. Masukkan API Key Google Maps Anda di sini
gmaps = googlemaps.Client(key=API_KEY)

# 2. Baca file excel input (sesuaikan nama file Anda)
input_file = 'kantor-nagari.xlsx' # Ganti dengan nama file excel Anda
df = pd.read_excel(input_file)

# Buat list kosong untuk menyimpan hasil koordinat
latitudes = []
longitudes = []

print("Memulai proses pencarian koordinat...")

# 3. Looping untuk mencari koordinat setiap tempat di kolom 'places'
for index, row in df.iterrows():
    place = row['places']
    try:
        # Panggil Geocoding API dari Google
        geocode_result = gmaps.geocode(place)
        
        # Jika hasilnya ada, ekstrak lat dan long
        if geocode_result:
            lat = geocode_result[0]['geometry']['location']['lat']
            lng = geocode_result[0]['geometry']['location']['lng']
            latitudes.append(lat)
            longitudes.append(lng)
            print(f"[{index+1}] Berhasil: {place} -> {lat}, {lng}")
        else:
            # Jika Google Maps tidak menemukan tempat tersebut
            latitudes.append(None)
            longitudes.append(None)
            print(f"[{index+1}] Gagal/Tidak ditemukan: {place}")
            
    except Exception as e:
        print(f"[{index+1}] Error pada {place}: {e}")
        latitudes.append(None)
        longitudes.append(None)
        
    # Jeda kecil agar tidak melampaui limit request Google Maps API per detik
    time.sleep(0.1)

# 4. Tambahkan kolom lat dan long ke DataFrame
df['lat'] = latitudes
df['long'] = longitudes

# 5. Simpan hasilnya ke file excel baru
output_file = 'hasil_koordinat_nagari.xlsx'
df.to_excel(output_file, index=False)

print(f"\nProses selesai! File telah disimpan sebagai: {output_file}")

Memulai proses pencarian koordinat...
[1] Berhasil: Kantor Wali Nagari Panti Selatan, Kab. Pasaman -> 0.3076844, 100.0902271
[2] Berhasil: Kantor Wali Nagari Simpang, Kab. Pasaman -> 0.0054237, 100.1559016
[3] Berhasil: Kantor Wali Nagari Padang Mantinggi, Kab. Pasaman -> 0.5640862999999999, 100.0152213
[4] Berhasil: Kantor Wali Nagari Panti, Kab. Pasaman -> 0.360623, 100.0607956
[5] Berhasil: Kantor Wali Nagari Ladang Panjang, Kab. Pasaman -> -0.0156442, 100.0810276
[6] Berhasil: Kantor Wali Nagari Pintu Padang, Kab. Pasaman -> 0.6160124, 100.1550955
[7] Berhasil: Kantor Wali Nagari Padang Galugua, Kab. Pasaman -> 0.4404232, 100.0486813
[8] Berhasil: Kantor Wali Nagari Tanjung Betung, Kab. Pasaman -> 0.4742571, 100.0432536
[9] Berhasil: Kantor Wali Nagari Koto Kaciak, Kab. Pasaman -> -0.0501787, 100.2057889
[10] Berhasil: Kantor Wali Nagari Lansek Kadok, Kab. Pasaman -> 0.519151, 100.0502456
[11] Berhasil: Kantor Wali Nagari Muaro Sungai Lolo, Kab. Pasaman -> 0.345801, 100.2967644
[12

## **Pembuatan titik-titik koordinat pencarian**

In [ ]:
# pip install geopandas

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

# 1. Konfigurasi
file_geojson = '13.08_Pasaman.geojson'
file_output = 'titik_pusat_150m_pasaman.xlsx'
radius_m = 150

# 2. Baca file GeoJSON
print("Membaca file GeoJSON...")
gdf = gpd.read_file(file_geojson)

# 3. Buat Proyeksi Metrik Lokal (Azimuthal Equidistant)
# Karena derajat Lat/Long tidak bisa dipakai untuk mengukur meter, 
# kita proyeksikan peta ke satuan meter berdasarkan titik tengah area tersebut.
titik_tengah = gdf.geometry.unary_union.centroid
custom_crs = f"+proj=aeqd +lat_0={titik_tengah.y} +lon_0={titik_tengah.x} +datum=WGS84 +units=m"
gdf_metric = gdf.to_crs(custom_crs)

# 4. Hitung batas (Bounding Box) dari area
minx, miny, maxx, maxy = gdf_metric.total_bounds

# 5. Parameter Hexagonal Grid untuk cakupan penuh
# Jarak antar titik pusat agar saling berpotongan menutupi semua area
dx = radius_m * np.sqrt(3)  # Jarak horizontal antar titik
dy = radius_m * 1.5         # Jarak vertikal antar baris

print("Membuat grid heksagonal...")
points = []
x_coords = np.arange(minx - dx, maxx + dx, dx)
y_coords = np.arange(miny - dy, maxy + dy, dy)

for row_idx, y in enumerate(y_coords):
    # Geser posisi X untuk baris ganjil agar membentuk pola heksagonal (sarang lebah)
    x_shift = (dx / 2.0) if (row_idx % 2 == 1) else 0.0
    for x in x_coords:
        points.append(Point(x + x_shift, y))

# Buat GeoDataFrame dari titik-titik grid
grid_gdf = gpd.GeoDataFrame(geometry=points, crs=custom_crs)

# 6. Filter titik yang hanya berada DI DALAM polygon Pasaman
print("Menyaring titik yang berada di dalam area Pasaman...")
# Menggunakan sjoin untuk mencari titik yang bersinggungan/berada di dalam polygon
titik_dalam_area = gpd.sjoin(grid_gdf, gdf_metric, how='inner', predicate='intersects')

# 7. Kembalikan proyeksi ke WGS84 (Derajat Lat/Long) standar GPS
titik_wgs84 = titik_dalam_area.to_crs(epsg=4326)

# 8. Ekstrak Latitude dan Longitude
print("Menyimpan ke Excel...")
df_output = pd.DataFrame({
    'No': range(1, len(titik_wgs84) + 1),
    'Latitude': titik_wgs84.geometry.y,
    'Longitude': titik_wgs84.geometry.x,
    'Radius_m': radius_m
})

# 9. Export ke Excel
df_output.to_excel(file_output, index=False)
print(f"Selesai! {len(df_output)} titik berhasil disimpan ke {file_output}")

Membaca file GeoJSON...


C:\Users\MYPC PRO L7V\AppData\Local\Temp\ipykernel_33904\57845275.py:18: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  titik_tengah = gdf.geometry.unary_union.centroid


Membuat grid heksagonal...
Menyaring titik yang berada di dalam area Pasaman...
Menyimpan ke Excel...
Selesai! 66749 titik berhasil disimpan ke titik_pusat_150m_pasaman.xlsx


Jika dirasa titik terlalu banyak bisa menggunakan strategi pendekatan lain, yaitu:
1. Memperluas radius titik supaya titiknya tidak terlalu banyak
2. Memberlakukan pengkategorian pada wilayah wilayah tertentu yang padat dengan usaha, misal di Kabupaten pasaman tepatnya di Nagari Pauah digolongkan sebagai tier I dikarenakan ramai, kemudian nagari panti selatan tidak terlalu ramai, sehingga dikelompokkan menjadi tier II, kemudian di kecamatan mapat tunggul tidak ada usaha yang ramai sehingga dikelompokkan menjadi tier III. Masing-masing tier memiliki radiusnya masing masing.
misal: 
- Tier I: 150 m 
- Tier II: 250 m
- Tier III: 350 m
<br>[Saya menggunakan strategi ini untuk menghemat credit]

In [4]:
gdf = gpd.read_file("pasaman-selected-area.geojson")
print(gdf)

     kd_propinsi kd_dati2 nm_dati2  tier  \
0            NaN     None     None     3   
1            NaN     None     None     2   
2           13.0       08  Pasaman     0   
3            NaN     None     None     3   
4            NaN     None     None     3   
..           ...      ...      ...   ...   
165          NaN     None     None     3   
166          NaN     None     None     3   
167          NaN     None     None     3   
168          NaN     None     None     3   
169          NaN     None     None     2   

                                              geometry  
0    POLYGON ((100.11145 0.33737, 100.1129 0.33809,...  
1    POLYGON ((100.10919 0.29904, 100.11201 0.29791...  
2    MULTIPOLYGON (((100.13318 -0.09487, 100.13325 ...  
3    POLYGON ((100.01252 0.66831, 100.01469 0.66961...  
4    POLYGON ((100.18305 0.69697, 100.18459 0.69932...  
..                                                 ...  
165  POLYGON ((100.16324 0.42617, 100.16083 0.42465...  
166  POLYGON ((

c:\Users\MYPC PRO L7V\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 2 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point

# radius per tier (meter) SESUAIKAN SENDIRI TERGANTUNG KEPADATAN USAHA DI SUATU WILAYAH
RADIUS_BY_TIER = {
    1: 200,
    2: 350,
    3: 500
}

def meters_to_deg_lat(m):
    return m / 111320

def meters_to_deg_lon(m, lat):
    return m / (111320 * np.cos(np.radians(lat)))

def generate_hex_points_in_polygon(polygon, radius_m):
    minx, miny, maxx, maxy = polygon.bounds

    center_lat = (miny + maxy) / 2

    dx = meters_to_deg_lon(2 * radius_m, center_lat)
    dy = meters_to_deg_lat(np.sqrt(3) * radius_m)

    points = []
    y = miny
    row = 0

    while y <= maxy:
        offset = dx / 2 if row % 2 == 1 else 0
        x = minx + offset

        while x <= maxx:
            p = Point(x, y)
            if polygon.contains(p):
                points.append((y, x))  # lat, lon
            x += dx

        y += dy
        row += 1

    return points

def generate_search_centers_from_geojson(geojson_path):
    gdf = gpd.read_file(geojson_path)

    results = []
    polygon_id = 0

    for idx, row in gdf.iterrows():
        tier = int(row["tier"])

        if tier not in [1, 2, 3]:
            continue

        radius = RADIUS_BY_TIER[tier]
        polygon = row.geometry

        points = generate_hex_points_in_polygon(polygon, radius)

        for lat, lon in points:
            results.append({
                "polygon_id": polygon_id,
                "tier": tier,
                "lat": lat,
                "lon": lon
            })

        polygon_id += 1

    return pd.DataFrame(results)


In [10]:
df = generate_search_centers_from_geojson("pasaman-selected-area.geojson")
print(df.head())

# simpan ke CSV
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"pasaman-with-centers_{timestamp}.xlsx"

df.to_excel(filename, index=False)

   polygon_id  tier       lat         lon
0           1     2  0.300230  100.112333
1           1     2  0.300230  100.118622
2           2     3  0.676086  100.010269
3           2     3  0.691645   99.992301
4           2     3  0.699425   99.987809


c:\Users\MYPC PRO L7V\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 2 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [ ]:
# pip install googlemaps

In [20]:
import pandas as pd
import googlemaps
import os
import time
import sys

# ================= Konfigurasi =================
API_KEY = API_KEY  # <-- GANTI INI DENGAN API KEY GOOGLE ANDA
INPUT_FILE = "pasaman-with-centers_20260420_112531.xlsx"  # Nama file Excel input
CACHE_FILE = "poi_cache.csv"              # File sementara untuk resume
FINAL_OUTPUT_FILE = "hasil_scraping_pasaman.csv"

# DICTIONARY TIER KE RADIUS (Dalam Meter)
# Sesuaikan radius untuk masing-masing tier di bawah ini, Harus sesuai dengan tier sebelumnya yang telah ditentukan radiusnya
TIER_RADIUS = {
    1: 200,  
    2: 350,  
    3: 500   
}
DEFAULT_RADIUS = 300 # Radius default jika nilai tier tidak ada di dictionary

# Inisialisasi Client Google Maps
gmaps = googlemaps.Client(key=API_KEY)

# Fungsi untuk membuat nama file unik (anti-replace)
def get_unique_filename(filename):
    base, extension = os.path.splitext(filename)
    counter = 1
    while os.path.exists(filename):
        filename = f"{base}-{counter}{extension}"
        counter += 1
    return filename

# ================= Setup Cache & Resume =================
processed_coords = set()

# Cek apakah cache sudah ada
if os.path.exists(CACHE_FILE):
    try:
        # Baca cache untuk mengetahui titik mana yang SUDAH selesai
        df_cache = pd.read_csv(CACHE_FILE)
        
        # Pastikan kolom referensi ada sebelum dibaca
        if 'ref_lat_awal' in df_cache.columns and 'ref_lon_awal' in df_cache.columns:
            for _, row in df_cache.iterrows():
                key = f"{row['ref_lat_awal']}_{row['ref_lon_awal']}"
                processed_coords.add(key)
            
            print(f"[INFO] Melanjutkan sesi... Ditemukan {len(processed_coords)} titik koordinat yang sudah diproses.")
    except Exception as e:
        print(f"[WARNING] Gagal membaca cache: {e}. Akan membuat cache baru jika diperlukan.")

# Buat header cache jika file belum ada
if not os.path.exists(CACHE_FILE):
    headers = pd.DataFrame(columns=[
        "place_id", "nama_poi", "type", "alamat", "vicinity_asli", 
        "status", "lat_poi", "long_poi", "ref_lat_awal", "ref_lon_awal", "tier_referensi", "radius_pencarian"
    ])
    headers.to_csv(CACHE_FILE, index=False)
    print("[INFO] File cache baru dibuat.")


# ================= Looping Utama =================

# Baca file input
if not os.path.exists(INPUT_FILE):
    print(f"[ERROR] File input '{INPUT_FILE}' tidak ditemukan!")
    sys.exit()

df_coords = pd.read_excel(INPUT_FILE)
total_data = len(df_coords)

print(f"[START] Mulai memproses {total_data} titik koordinat...")

for index, row in df_coords.iterrows():
    lat = row['lat']
    lon = row['lon']
    tier = row['tier']
    coord_key = f"{lat}_{lon}" # Kunci unik untuk pengecekan

    # Ambil radius berdasarkan tier dari dictionary
    current_radius = TIER_RADIUS.get(tier, DEFAULT_RADIUS)

    # 1. SKIP JIKA SUDAH ADA DI CACHE
    if coord_key in processed_coords:
        continue

    print(f"\n[{index + 1}/{total_data}] Memproses Lat: {lat}, Lon: {lon} | Tier: {tier} | Radius: {current_radius}m")
    
    page_count = 1
    next_page_token = None
    
    # 2. LOOPING PAGINATION
    while True:
        try:
            # Jika ada token halaman berikutnya
            if next_page_token:
                print("  -> Menunggu token halaman berikutnya (2 detik)...")
                time.sleep(2) # Wajib jeda agar token valid di server Google
                places_result = gmaps.places_nearby(
                    page_token=next_page_token
                )
            # Jika ini halaman pertama
            else:
                places_result = gmaps.places_nearby(
                    location=(lat, lon),
                    radius=current_radius
                )
        except googlemaps.exceptions.ApiError as e:
            # Tangkap error spesifik dari Google Maps API (seperti OVER_QUERY_LIMIT)
            status = str(e)
            if "OVER_QUERY_LIMIT" in status:
                print("[CRITICAL] Kuota API Habis! Berhenti.")
                sys.exit()
            else:
                print(f"  [API ERROR] {e}. Lewati halaman ini.")
                break
        except Exception as e:
            print(f"  [ERROR KONEKSI/SISTEM] {e}. Lewati titik ini.")
            break 
            
        results = places_result.get("results", [])
        poi_list = []
        
        # Ekstrak Data
        for place in results:
            # Logika Alamat Lengkap (Compound Code)
            alamat_display = place.get("vicinity")
            plus_code = place.get("plus_code", {})
            if "compound_code" in plus_code:
                alamat_display = plus_code["compound_code"]

            poi_list.append({
                "place_id": place.get("place_id"),
                "nama_poi": place.get("name"),
                "type": ", ".join(place.get("types", [])),
                "alamat": alamat_display,
                "vicinity_asli": place.get("vicinity"), 
                "status": place.get("business_status", "UNKNOWN"),
                "lat_poi": place["geometry"]["location"]["lat"],
                "long_poi": place["geometry"]["location"]["lng"],
                "ref_lat_awal": lat,
                "ref_lon_awal": lon,
                "tier_referensi": tier,
                "radius_pencarian": current_radius
            })
            
        # Jika Kosong (Zero Results) di Halaman 1
        if not poi_list and page_count == 1:
            # Simpan dummy record agar titik ini terekam "sudah diproses"
            poi_list.append({
                "place_id": "NONE", "nama_poi": "TIDAK ADA POI", 
                "ref_lat_awal": lat, "ref_lon_awal": lon,
                "tier_referensi": tier, "radius_pencarian": current_radius
            })
            print("  -> Tidak ada POI ditemukan (ZERO_RESULTS).")

        # Simpan ke CSV Cache (Append Mode)
        if poi_list:
            df_temp = pd.DataFrame(poi_list)
            # Pastikan urutan kolom sesuai header dengan mengindeks ulang kolom
            cols = ["place_id", "nama_poi", "type", "alamat", "vicinity_asli", 
                    "status", "lat_poi", "long_poi", "ref_lat_awal", "ref_lon_awal", "tier_referensi", "radius_pencarian"]
            
            # Isi kolom yang mungkin NaN (seperti type, alamat, dll untuk record 'NONE') dengan nilai kosong
            for col in cols:
                if col not in df_temp.columns:
                    df_temp[col] = ""
                    
            df_temp = df_temp[cols]
            df_temp.to_csv(CACHE_FILE, mode='a', header=False, index=False)
            
            if results:
                print(f"  -> Disimpan {len(results)} POI (Halaman {page_count})")
            
        # Cek Next Page Token dari respon dictionary
        next_page_token = places_result.get("next_page_token")
        
        if next_page_token:
            page_count += 1
        else:
            break # Selesai untuk titik ini


# ================= Finalisasi & Bersih-bersih Data =================
print("\n[INFO] Semua proses penarikan data selesai!")

if os.path.exists(CACHE_FILE):
    # Baca seluruh data dari cache
    final_df = pd.read_csv(CACHE_FILE)

    # Buang baris "Dummy" (ZERO_RESULTS) yang kita pakai untuk tracking cache
    final_df = final_df[final_df['place_id'] != 'NONE']

    # Buang duplikat berdasarkan place_id (karena area radius mungkin tumpang tindih)
    final_df = final_df.drop_duplicates(subset=['place_id'])

    # Simpan ke file final yang unik
    final_output_name = get_unique_filename(FINAL_OUTPUT_FILE)
    final_df.to_csv(final_output_name, index=False)

    print(f"[SUKSES] Data dibersihkan dari duplikat. Total POI Unik: {len(final_df)}")
    print(f"[SUKSES] File final disimpan dengan nama: {final_output_name}")
else:
    print("[INFO] Tidak ada data yang dihasilkan.")

[INFO] File cache baru dibuat.
[START] Mulai memproses 9 titik koordinat...

[1/9] Memproses Lat: 0.342038882976302, Lon: 100.112393989122 | Tier: 3.0 | Radius: 500m
  -> Disimpan 8 POI (Halaman 1)

[2/9] Memproses Lat: 0.299451934380831, Lon: 100.111884269365 | Tier: 2.0 | Radius: 350m
  -> Disimpan 5 POI (Halaman 1)

[3/9] Memproses Lat: 0.299451934380831, Lon: 100.117274209467 | Tier: 2.0 | Radius: 350m
  -> Disimpan 20 POI (Halaman 1)
  -> Menunggu token halaman berikutnya (2 detik)...
  -> Disimpan 5 POI (Halaman 2)

[4/9] Memproses Lat: 0.672974223487065, Lon: 100.008472248511 | Tier: 3.0 | Radius: 500m
  -> Disimpan 1 POI (Halaman 1)

[5/9] Memproses Lat: 0.672974223487065, Lon: 100.0138625574 | Tier: 3.0 | Radius: 500m
  -> Tidak ada POI ditemukan (ZERO_RESULTS).

[6/9] Memproses Lat: 0.571040974044356, Lon: 100.15092266165 | Tier: 2.0 | Radius: 350m
  -> Disimpan 1 POI (Halaman 1)

[7/9] Memproses Lat: 0.393171091248324, Lon: 100.110143247518 | Tier: 1.0 | Radius: 200m
  -> Ti

In [ ]:
import pandas as pd
import requests
import time
import os
import sys

# ================= KONFIGURASI =================
API_KEY = "MASUKKAN_API_KEY_ANDA_DISINI"
INPUT_FILE = "data_input.xlsx"  # File Excel input Anda
OUTPUT_FILE = "hasil_poi_pasaman_progress.csv" # File output (sekaligus cache)
RADIUS = 300  # Dalam meter

# ================= FUNGSI UTAMA =================

def get_google_places(lat, lon, api_key, radius):
    """
    Mengambil data POI dengan dukungan pagination (next_page_token).
    Mengembalikan list of dictionaries.
    """
    all_results = []
    base_url = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    
    params = {
        "location": f"{lat},{lon}",
        "radius": radius,
        "key": api_key
    }
    
    while True:
        try:
            response = requests.get(base_url, params=params, timeout=10)
            data = response.json()
            
            if data.get('status') == 'OK':
                results = data.get('results', [])
                for place in results:
                    place_data = {
                        "place_id": place.get("place_id"),
                        "nama_poi": place.get("name"),
                        "type": ", ".join(place.get("types", [])),
                        "alamat": place.get("vicinity"),
                        "status": place.get("business_status", "UNKNOWN"),
                        "lat_poi": place["geometry"]["location"]["lat"],
                        "long_poi": place["geometry"]["location"]["lng"],
                        "ref_lat_awal": lat, # Penting untuk tracking resume
                        "ref_lon_awal": lon
                    }
                    all_results.append(place_data)
            
            elif data.get('status') == 'ZERO_RESULTS':
                break # Tidak ada POI di sini, lanjut.
            elif data.get('status') == 'OVER_QUERY_LIMIT':
                print("\n[CRITICAL] Kuota API habis atau Request terlalu cepat!")
                sys.exit() # Berhenti total
            else:
                print(f"Warning: API Status {data.get('status')}")
                break

            # Cek Pagination
            next_page_token = data.get('next_page_token')
            if next_page_token:
                # PENTING: Google butuh waktu jeda sebelum token valid
                time.sleep(2) 
                params = {
                    "pagetoken": next_page_token,
                    "key": api_key
                }
            else:
                break # Tidak ada halaman selanjutnya
                
        except Exception as e:
            print(f"Error koneksi: {e}")
            break

    return all_results

# ================= EKSEKUSI =================

def main():
    # 1. Baca Data Input
    if not os.path.exists(INPUT_FILE):
        print(f"File input {INPUT_FILE} tidak ditemukan!")
        return
    
    df_input = pd.read_excel(INPUT_FILE)
    total_input = len(df_input)
    print(f"Total data input: {total_input} baris.")

    # 2. Logika Resume / Cache
    processed_coords = set()
    file_exists = os.path.isfile(OUTPUT_FILE)
    
    if file_exists:
        print(f"File output '{OUTPUT_FILE}' ditemukan. Mengecek progress...")
        try:
            # Baca CSV yang sudah ada untuk melihat koordinat mana yang selesai
            df_existing = pd.read_csv(OUTPUT_FILE)
            # Buat set kombinasi lat,lon yang sudah ada (supaya unik)
            # Kita gunakan string f"{lat}_{lon}" sebagai key sederhana
            if 'ref_lat_awal' in df_existing.columns and 'ref_lon_awal' in df_existing.columns:
                processed_coords = set(
                    zip(df_existing['ref_lat_awal'].astype(str), df_existing['ref_lon_awal'].astype(str))
                )
            print(f"Melanjutkan progress: {len(processed_coords)} titik koordinat sudah diproses sebelumnya.")
        except Exception as e:
            print(f"Gagal membaca file progress: {e}. Akan mulai dari awal (hati-hati overwrite).")

    # 3. Looping Data
    mode = 'a' if file_exists else 'w'
    header = not file_exists # Tulis header hanya jika file baru dibuat

    count_processed = 0
    
    for index, row in df_input.iterrows():
        lat = row['lat']
        lon = row['lon']
        
        # Cek apakah koordinat ini sudah diproses
        coord_key = (str(lat), str(lon))
        if coord_key in processed_coords:
            continue # SKIP, lanjut ke baris berikutnya
        
        print(f"[{index+1}/{total_input}] Mengambil data untuk: {lat}, {lon} ... ", end="", flush=True)
        
        # Panggil API
        results = get_google_places(lat, lon, API_KEY, RADIUS)
        
        # Simpan Hasil Langsung (Batch per koordinat)
        if results:
            df_batch = pd.DataFrame(results)
            # Simpan ke CSV mode append
            df_batch.to_csv(OUTPUT_FILE, mode='a', header=header, index=False)
            header = False # Setelah penulisan pertama, header tidak perlu lagi
            print(f"Dapat {len(results)} POI. Disimpan.")
        else:
            # Jika tidak ada hasil, kita tetap perlu mencatat bahwa titik ini sudah diproses
            # Opsional: Bisa buat log terpisah, tapi agar simple kita biarkan saja
            # atau simpan dummy record jika ingin benar-benar strict trackingnya.
            print("Tidak ada POI ditemukan.")

        count_processed += 1
        
        # Opsional: Sleep kecil agar tidak spam request berlebihan
        time.sleep(0.5)

    print("\n================ SELESAI ================")
    print(f"Proses selesai. Data tersimpan di {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

In [15]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

def petakan_tanpa_nama_kecamatan(file_excel_input, file_geojson, file_excel_output):
    # 1. Baca File Excel (Data Input)
    print("Membaca file Excel input...")
    df_input = pd.read_csv(file_excel_input)
    
    # 2. Buat Geometri dari Lat/Long
    # Point(x, y) -> (long, lat)
    geometry = [Point(xy) for xy in zip(df_input['long_poi'], df_input['lat_poi'])]
    gdf_points = gpd.GeoDataFrame(df_input, geometry=geometry, crs="EPSG:4326")
    
    # 3. Baca File GeoJSON
    print("Membaca file GeoJSON...")
    gdf_regions = gpd.read_file(file_geojson)
    
    # Pastikan CRS sama
    if gdf_regions.crs != gdf_points.crs:
        gdf_regions = gdf_regions.to_crs(gdf_points.crs)
        
    # 4. Spatial Join (Left Join)
    # Gunakan left join agar data yang tidak punya wilayah (koordinat luar) tidak hilang
    print("Melakukan pencocokan lokasi...")
    joined_gdf = gpd.sjoin(gdf_points, gdf_regions, how="left", predicate="within")
    
    # 5. Membuat Kode Lengkap (Gabungan String)
    # Cek ketersediaan kolom kode di GeoJSON
    cols_check = ['kd_propinsi', 'kd_dati2', 'kd_kecamatan', 'kd_kelurahan']
    
    if all(col in joined_gdf.columns for col in cols_check):
        # Buat Kode Kecamatan (Gabungan Propinsi + Kab/Kota + Kecamatan)
        joined_gdf['Kode_Kecamatan'] = (
            joined_gdf['kd_propinsi'].astype(str) + 
            joined_gdf['kd_dati2'].astype(str) + 
            joined_gdf['kd_kecamatan'].astype(str)
        )
        
        # Buat Kode Desa (Gabungan Kode Kecamatan + Kelurahan)
        joined_gdf['Kode_Desa'] = (
            joined_gdf['Kode_Kecamatan'] + 
            joined_gdf['kd_kelurahan'].astype(str)
        )
    else:
        # Jika kolom kode di GeoJSON tidak lengkap/namanya beda, isi kosong
        print("Warning: Kolom kode wilayah tidak lengkap di GeoJSON.")
        joined_gdf['Kode_Kecamatan'] = ""
        joined_gdf['Kode_Desa'] = ""

    # 6. Ambil Nama Desa
    # Rename 'nm_kelurahan' jadi 'Nama_Desa' agar lebih jelas
    if 'nm_kelurahan' in joined_gdf.columns:
        joined_gdf.rename(columns={'nm_kelurahan': 'Nama_Desa'}, inplace=True)
    else:
        joined_gdf['Nama_Desa'] = ""

    # 7. Bersihkan Kolom yang Tidak Diperlukan
    # Kita hapus kolom geometry, index sjoin, dan pecahan kode aslinya
    cols_to_drop = ['geometry', 'index_right', 'kd_propinsi', 'kd_dati2', 'kd_kecamatan', 'kd_kelurahan']
    
    # Kita juga pastikan menghapus kolom nama kecamatan jika ada di GeoJSON (sesuai request)
    potential_kec_names = ['nm_kecamatan', 'nama_kecamatan', 'WADMKC']
    cols_to_drop.extend(potential_kec_names)
    
    # Eksekusi penghapusan kolom (hanya jika kolom tersebut ada)
    joined_gdf.drop(columns=[c for c in cols_to_drop if c in joined_gdf.columns], inplace=True)

    # 8. Simpan Hasil
    print(f"Menyimpan ke {file_excel_output}...")
    joined_gdf.to_excel(file_excel_output, index=False)
    print("Selesai.")

# --- PENGGUNAAN ---
file_input = 'hasil_poi_pasaman-8.csv'       # Ganti dengan file Excel Anda
file_json = '13.08_kelurahan.geojson'   # Ganti dengan file GeoJSON Anda
file_output = 'hasil_mapping.xlsx'

# Jalankan fungsi
petakan_tanpa_nama_kecamatan(file_input, file_json, file_output)

Membaca file Excel input...
Membaca file GeoJSON...
Melakukan pencocokan lokasi...
Menyimpan ke hasil_mapping.xlsx...
Selesai.


In [5]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# --- PENGATURAN NAMA FILE ---
excel_file = 'HASIL_SCRAPING_GOOGLE_MAPS_NEW.xlsx'          # Ganti dengan nama file Excel Anda
geojson_file = '13.08_kelurahan.geojson'    # File batas wilayah (sesuai gambar)
output_file = 'hasil_klasifikasi_desa_NEW.xlsx' # Nama file output
# ----------------------------

def klasifikasi_wilayah():
    try:
        print("1. Membaca data Excel...")
        df_poi = pd.read_excel(excel_file)
        
        # Pastikan kolom lat_poi dan long_poi ada (ubah tipe ke float jika perlu)
        df_poi['lat_poi'] = df_poi['lat_poi'].astype(float)
        df_poi['long_poi'] = df_poi['long_poi'].astype(float)

        # 2. Membuat Point Geometry (Perhatikan urutannya: Longitude (X), Latitude (Y))
        print("2. Membuat data geospasial dari koordinat...")
        geometry = [Point(xy) for xy in zip(df_poi['long_poi'], df_poi['lat_poi'])]
        
        # Buat GeoDataFrame dan set CRS ke EPSG:4326 (Sistem koordinat standar GPS/WGS84)
        gdf_poi = gpd.GeoDataFrame(df_poi, geometry=geometry, crs="EPSG:4326")

        print("3. Membaca batas wilayah GeoJSON...")
        gdf_wilayah = gpd.read_file(geojson_file)

        # Pastikan sistem koordinat (CRS) kedua data sama sebelum digabung
        if gdf_wilayah.crs != gdf_poi.crs:
            gdf_wilayah = gdf_wilayah.to_crs(gdf_poi.crs)

        print("4. Melakukan pencocokan (Spatial Join)...")
        # how='left': Mempertahankan semua baris Excel meskipun koordinatnya meleset di luar batas polygon
        # predicate='within': Mencari titik yang berada di dalam polygon wilayah
        hasil_join = gpd.sjoin(gdf_poi, gdf_wilayah, how="left", predicate="within")

        # 5. Merapikan kolom output
        # Mengambil semua kolom asli dari Excel
        kolom_asli = list(df_poi.columns)
        # Menambahkan kolom target dari GeoJSON sesuai permintaan
        kolom_tambahan = ['nm_kelurahan', 'kd_kelurahan', 'kd_kecamatan', 'kd_dati2', 'kd_propinsi']
        
        kolom_final = kolom_asli + kolom_tambahan
        
        # Filter dataframe hanya pada kolom yang dibutuhkan
        df_final = hasil_join[kolom_final]

        print("5. Menyimpan hasil ke Excel...")
        df_final.to_excel(output_file, index=False)
        print(f"Selesai! Data berhasil disimpan di: {output_file}")

    except Exception as e:
        print(f"Terjadi kesalahan: {e}")

# Jalankan fungsi
klasifikasi_wilayah()

1. Membaca data Excel...
2. Membuat data geospasial dari koordinat...
3. Membaca batas wilayah GeoJSON...
4. Melakukan pencocokan (Spatial Join)...
5. Menyimpan hasil ke Excel...
Selesai! Data berhasil disimpan di: hasil_klasifikasi_desa_NEW.xlsx


In [3]:
import json
import pandas as pd

# --- PENGATURAN FILE ---
input_json = 'C:/Users/MYPC PRO L7V/Downloads/FAIS/Work/SBR/scraping-maps/kode/kec_pasaman.json'  # Ganti dengan nama file JSON Anda
output_excel = 'C:/Users/MYPC PRO L7V/Downloads/FAIS/Work/SBR/scraping-maps/kode/tabel_kecamatan_desa.xlsx' # Nama file Excel hasil output
# -----------------------

def konversi_json_ke_excel(file_json, file_excel):
    try:
        # 1. Membaca data JSON
        with open(file_json, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # List untuk menyimpan baris data sebelum dijadikan DataFrame
        baris_data = []

        # 2. Iterasi untuk setiap kecamatan
        for kecamatan in data:
            # Ambil data level kecamatan
            id_kab_kota = kecamatan.get('kabupaten_kota_id')
            nama_kecamatan = kecamatan.get('nama')
            id_kecamatan = kecamatan.get('id')
            
            # Cek apakah kecamatan memiliki daftar desa
            if 'desa' in kecamatan and isinstance(kecamatan['desa'], list):
                # 3. Iterasi untuk setiap desa di dalam kecamatan tersebut
                for desa in kecamatan['desa']:
                    id_desa = desa.get('id')
                    nama_desa = desa.get('nama')
                    
                    # Tambahkan data sebagai satu baris ke dalam list
                    baris_data.append({
                        'id_kab_kota': id_kab_kota,
                        'nama_kecamatan': nama_kecamatan,
                        'id_kecamatan': id_kecamatan,
                        'id_desa': id_desa,
                        'nama_desa': nama_desa
                    })
            else:
                # Jika tidak ada desa terdaftar, tetap masukkan kecamatannya 
                # namun dengan id_desa dan nama_desa kosong
                baris_data.append({
                    'id_kab_kota': id_kab_kota,
                    'nama_kecamatan': nama_kecamatan,
                    'id_kecamatan': id_kecamatan,
                    'id_desa': None,
                    'nama_desa': None
                })

        # 4. Membuat Pandas DataFrame dari list dictionary
        df = pd.DataFrame(baris_data)

        # 5. Menyimpan DataFrame ke file Excel
        df.to_excel(file_excel, index=False)
        print(f"Berhasil! Data telah diekspor ke: {file_excel}")

    except FileNotFoundError:
        print(f"Error: File '{file_json}' tidak ditemukan. Pastikan nama dan lokasinya benar.")
    except json.JSONDecodeError:
        print(f"Error: Format file '{file_json}' bukan JSON yang valid.")
    except Exception as e:
        print(f"Terjadi kesalahan yang tidak terduga: {e}")

# Jalankan fungsi
konversi_json_ke_excel(input_json, output_excel)

Berhasil! Data telah diekspor ke: C:/Users/MYPC PRO L7V/Downloads/FAIS/Work/SBR/scraping-maps/kode/tabel_kecamatan_desa.xlsx
